# Supabase Transaction Pooler to S3 CSV export

This notebook reads configured Supabase tables through Transaction Pooler (`:6543`) and writes versioned CSV snapshots to S3. Credentials stay in `.env` or the environment, never in the notebook.

## One-time setup

From the project root:

```bash
python3 -m pip install -r requirements-dev.txt
cp .env.example .env
# Fill in the transaction-pooler URL, S3 bucket, and selected tables in .env
```

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.export_pipeline import Settings, export_all_tables

settings = Settings.from_env()
print(f'S3 destination: s3://{settings.bucket}/{settings.prefix}/')
print('Tables:', [f'{item.schema}.{item.table}' for item in settings.tables])


S3 destination: s3://mlops-01-bucket/raw_data/
Tables: ['public.enrollees', 'public.enrollee_education', 'public.work_experience', 'public.training_hours', 'public.employment', 'public.city_dev_index']


In [2]:
# Starts SELECT -> CSV -> S3 upload for every configured table.
report = export_all_tables(settings)
report


{'run_id': '20260930T082115Z',
 'exported_at': '2026-09-30T08:21:40.121705+00:00',
 'source': 'supabase-transaction-pooler',
 'tables': [{'schema': 'public',
   'table': 'enrollees',
   'primary_key': 'enrollee_id',
   'rows': 19158,
   's3_uri': 's3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/enrollees/data.csv',
   'sha256': '754b523112e330f00de0398e2b6ac7c84f0671047c0be562443c4a8498504ce6'},
  {'schema': 'public',
   'table': 'enrollee_education',
   'primary_key': 'enrollee_id',
   'rows': 19158,
   's3_uri': 's3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/enrollee_education/data.csv',
   'sha256': 'e4f4dafc2de65083ceafa1c2d2c3b96ee6f850488dec50fdb0cca38887de88d6'},
  {'schema': 'public',
   'table': 'work_experience',
   'primary_key': 'enrollee_id',
   'rows': 19158,
   's3_uri': 's3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/work_experience/data.csv',
   'sha256': '28f7e5232558b5d3a63ca0fbba1f8d795415284bd50d50896494da6728098863'},
  {'

In [3]:
# Compact result table for copying into a runbook or checking the export.
for item in report['tables']:
    print(f"{item['schema']}.{item['table']}: {item['rows']:,} rows -> {item['s3_uri']}")
print('Manifest:', report['manifest_s3_uri'])


public.enrollees: 19,158 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/enrollees/data.csv
public.enrollee_education: 19,158 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/enrollee_education/data.csv
public.work_experience: 19,158 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/work_experience/data.csv
public.training_hours: 19,158 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/training_hours/data.csv
public.employment: 19,158 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/employment/data.csv
public.city_dev_index: 123 rows -> s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/public/city_dev_index/data.csv
Manifest: s3://mlops-01-bucket/raw_data/run_id=20260930T082115Z/manifest.json
